# <h1>MultiIndex / advanced indexing</h1>

<p>This section covers <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-hierarchical">indexing with a MultiIndex</a> and <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-index-types">other advanced indexing features</a>.</p>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/indexing.html#indexing">Indexing and Selecting Data</a> for general indexing documentation.</p>

<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>Whether a copy or a reference is returned for a setting operation may
depend on the context.  This is sometimes called <code>chained assignment</code> and should be avoided.
See <a href="https://pandas.pydata.org/docs/user_guide/indexing.html#indexing-view-versus-copy">Returning a View versus Copy</a>.</p>
</div>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-selection">cookbook</a> for some advanced strategies.</p>

## <h2>Hierarchical indexing (MultiIndex)</h2>

<p>Hierarchical / Multi-level indexing is very exciting as it opens the door to some quite sophisticated data analysis and manipulation, especially for working with higher dimensional data.
In essence, it enables you to store and manipulate data with an arbitrary number of dimensions in lower dimensional data structures like <code>Series</code> (1d) and <code>DataFrame</code> (2d).</p>

<p>In this section, we will show what exactly we mean by “hierarchical” indexing and how it integrates with all of the pandas indexing functionality described above and in prior sections.
Later, when discussing <a href="https://pandas.pydata.org/docs/user_guide/groupby.html#groupby">group by</a> and <a href="https://pandas.pydata.org/docs/user_guide/reshaping.html#reshaping"><span class="std std-ref">pivoting and reshaping data</a>, we’ll show non-trivial applications to illustrate how it aids in structuring data for analysis.</p>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-multi-index">cookbook</a> for some advanced strategies.</p>

### <h3>Creating a MultiIndex (hierarchical index) object</h3>

<p>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.html#pandas.MultiIndex" title="pandas.MultiIndex"><code>MultiIndex</code></a> object is the hierarchical analogue of the standard
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html#pandas.Index" title="pandas.Index"><code>Index</code></a> object which typically stores the axis labels in pandas objects.
You can think of <code>MultiIndex</code> as an array of tuples where each tuple is unique.
A <code>MultiIndex</code> can be created from a list of arrays (using
<a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_arrays.html#pandas.MultiIndex.from_arrays" title="pandas.MultiIndex.from_arrays"><code>MultiIndex.from_arrays()</code></a>), an array of tuples (using
<a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_tuples.html#pandas.MultiIndex.from_tuples" title="pandas.MultiIndex.from_tuples"><code>MultiIndex.from_tuples()</code></a>), a crossed set of iterables (using
<a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_product.html#pandas.MultiIndex.from_product" title="pandas.MultiIndex.from_product"><code>MultiIndex.from_product()</code></a>), or a <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html#pandas.DataFrame" title="pandas.DataFrame"><code>DataFrame</code></a> (using <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_frame.html#pandas.MultiIndex.from_frame" title="pandas.MultiIndex.from_frame"><code>MultiIndex.from_frame()</code></a>).
The <code>Index</code> constructor will attempt to return a <code>MultiIndex</code> when it is passed a list of tuples.
The following examples demonstrate different ways to initialize MultiIndexes.</p>

In [1]:
import pandas as pd
import numpy as np

In [2]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

In [3]:
tuples = list(zip(*arrays))

In [4]:
tuples

[('bar', 'one'),
 ('bar', 'two'),
 ('baz', 'one'),
 ('baz', 'two'),
 ('foo', 'one'),
 ('foo', 'two'),
 ('qux', 'one'),
 ('qux', 'two')]

In [5]:
index  =  pd.MultiIndex.from_tuples(tuples, names = ["first", "second"])

In [6]:
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [7]:
s = pd.Series(np.random.randn(8), index=index)

In [8]:
s

first  second
bar    one      -1.237450
       two       0.183446
baz    one      -0.318575
       two      -0.313478
foo    one       0.589356
       two       1.772349
qux    one       0.419610
       two      -0.220375
dtype: float64

<p>When you want every pairing of the elements in two iterables, it can be easier to use the <a href="../reference/api/pandas.MultiIndex.from_product.html#pandas.MultiIndex.from_product" title="pandas.MultiIndex.from_product"><code>MultiIndex.from_product()</code></a> method:</p>

In [10]:
iterables  =  [["bar", "baz", "foo", "qux" ], ["one", "two"]]

In [11]:
pd.MultiIndex.from_product(iterables, names=["first", "second"])

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

<p>You can also construct a <code>MultiIndex</code> from a <code>DataFrame</code> directly, using the method <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_frame.html#pandas.MultiIndex.from_frame" title="pandas.MultiIndex.from_frame"><code>MultiIndex.from_frame()</code></a>.
This is a complementary method to <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.to_frame.html#pandas.MultiIndex.to_frame" title="pandas.MultiIndex.to_frame"><code>MultiIndex.to_frame()</code></a>.</p>

In [12]:
df = pd.DataFrame(
    [["bar", "one"], ["bar", "two"], ["foo", "one"], ["foo", "two"]],
    columns=["first", "second"],
)

In [13]:
pd.MultiIndex.from_frame(df)

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('foo', 'one'),
            ('foo', 'two')],
           names=['first', 'second'])

<p>As a convenience, you can pass a list of arrays directly into <code>Series</code> or <code>DataFrame</code> to construct a <code>MultiIndex</code> automatically:</p>

In [14]:
arrays = [
    np.array(["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"]),
    np.array(["one", "two", "one", "two", "one", "two", "one", "two"]),
]

In [15]:
s = pd.Series(np.random.randn(8), index=arrays)

In [16]:
s

bar  one   -0.814359
     two    0.408115
baz  one   -0.797532
     two   -2.399718
foo  one   -1.238620
     two   -0.171019
qux  one   -0.228111
     two    0.320595
dtype: float64

In [17]:
df = pd.DataFrame(np.random.randn(8, 4), index=arrays)

In [18]:
df

0         1         2         3
bar one  1.009346  0.603782 -0.467420 -0.300018
    two  0.706239  0.016150 -0.127963  0.698338
baz one  0.076603  0.347282  0.729846 -0.160353
    two -0.955747 -1.186058 -0.599815 -1.074470
foo one  0.894275  0.452607  1.668248  1.195994
    two  0.497798  0.173207  2.283516  1.463183
qux one  0.975411  0.148113 -0.296275 -2.009371
    two  0.758621  1.056466 -1.038671 -0.018772

<p>All of the <code>MultiIndex</code> constructors accept a <code>names</code> argument which stores string names for the levels themselves.
If no names are provided, <code>None</code> will be assigned:</p>

In [19]:
df.index.names

FrozenList([None, None])

<p>This index can back any axis of a pandas object, and the number of <strong>levels</strong> of the index is up to you:</p>

In [20]:
df = pd.DataFrame(np.random.randn(3, 8), index=["A", "B", "C"], columns=index)

In [21]:
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A      -0.245699 -0.919419 -0.846182  0.055406  0.661378 -0.290777 -1.467747   
B      -0.654189  0.057187  1.570267  0.124010 -0.419803 -0.372840 -0.368978   
C       0.208356  0.347714  1.380485  0.533335 -0.279130 -0.957582 -0.303195   

first             
second       two  
A       0.743368  
B      -0.567214  
C      -0.038306

In [22]:
pd.DataFrame(np.random.randn(6, 6), index=index[:6], columns=index[:6])

first              bar                 baz                 foo          
second             one       two       one       two       one       two
first second                                                            
bar   one     1.075732 -1.002769 -0.036811  0.333032 -0.019484 -0.610955
      two    -0.795157 -0.367439 -0.117217 -1.324374 -0.431988  0.647391
baz   one     1.244690  1.327118 -1.186815  0.635033 -1.346094 -0.860929
      two     1.399641  0.361146  1.249055  0.155434  0.316307 -0.447675
foo   one     0.065122  0.211881  0.570239 -1.401055  0.480254 -1.238013
      two    -0.070201  0.030434 -0.193962 -0.473481 -0.364655 -0.944337

<p>We’ve “sparsified” the higher levels of the indexes to make the console output a bit easier on the eyes.
Note that how the index is displayed can be controlled using the
<code>multi_sparse</code> option in <code>pandas.set_options()</code>:</p>

In [23]:
with pd.option_context("display.multi_sparse", False):
    df

<p>It’s worth keeping in mind that there’s nothing preventing you from using tuples as atomic labels on an axis:</p>

In [24]:
pd.Series(np.random.randn(8), index=tuples)

,0
"(bar, one)",-0.407579
"(bar, two)",0.839451
"(baz, one)",-2.061620
"(baz, two)",0.488152
"(foo, one)",0.388456
"(foo, two)",-1.485168
"(qux, one)",0.922623
"(qux, two)",-0.749783


<p>The reason that the <code>MultiIndex</code> matters is that it can allow you to do grouping, selection, and reshaping operations as we will describe below and in subsequent areas of the documentation.
As you will see in later sections, you can find yourself working with hierarchically-indexed data without creating a <code>MultiIndex</code> explicitly yourself.
However, when loading data from a file, you may wish to generate your own <code>MultiIndex</code> when preparing the data set.</p>